# Hardware 4-qubit scheduling-policy comparison

This notebook runs a 4-qubit Heisenberg workload on real QUBEX hardware and compares the modern Qiskit-timed execution path with the deprecated `legacy_device_gateway` timing policy.

The workload uses all labels in `bell_state.DEFAULT_QUBIT_LABELS`, so the default target is `Q036,Q037,Q038,Q039` on `144Qv2`. The Heisenberg edges are derived from the generated topology by taking one direction for each calibrated undirected pair.

The same circuit family is used for three curves:

1. ideal local simulation,
2. QUBEX hardware with `timing_policy="qiskit"` and ALAP scheduling,
3. QUBEX hardware with `timing_policy="legacy_device_gateway"`.

By default this notebook connects to hardware. Set `RUN_ON_HARDWARE = False` if you only want compilation, schedule validation, and the ideal simulation.

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import json
import sys
from collections import deque
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import SVG, display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector

for module_name in (
    "qiskit_qubex_provider.executor",
    "qiskit_qubex_provider.backend",
    "qiskit_qubex_provider.provider",
    "qiskit_qubex_provider",
):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

from qiskit_qubex_provider import (
    QubexProvider,
    build_pulse_schedule_timeline_figure,
    diff_pulse_schedules,
    summarize_pulse_schedule,
)

HERE = Path.cwd()
if not (HERE / "bell_state.py").exists():
    HERE = Path("examples/hardware").resolve()

spec = importlib.util.spec_from_file_location("hardware_bell_state", HERE / "bell_state.py")
bell_state = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(bell_state)


In [ ]:
DEVICE_ID = bell_state.DEFAULT_DEVICE_ID
QUBIT_LABELS = bell_state.DEFAULT_QUBIT_LABELS
BELL_PAIR = bell_state.DEFAULT_BELL_PAIR
CONFIG_ROOT = HERE / "qubex-config"
OUTPUT_DIR = HERE / "generated"

if len(QUBIT_LABELS) != 4:
    raise ValueError(f"scheduling_comparison_4q requires exactly 4 qubits, got {QUBIT_LABELS!r}")

NUM_QUBITS = len(QUBIT_LABELS)
SHOTS = 4096
RUN_ON_HARDWARE = True
READOUT_MITIGATION = True

N_STEPS = 1
TOTAL_TIMES = np.linspace(0, 5, 30)
TOTAL_TIMES_IDEAL = np.linspace(0, 5, 241)
SAMPLE_TOTAL_TIME = 2.5
REFERENCE_J_VALUES = np.array([
    0.542641286533492,
    0.5345927977051017,
    -0.958496101281197,
])
REFERENCE_ANGLE_SCALE = 0.5

TOPOLOGY_PATH = bell_state.topology_json_path(
    output_dir=OUTPUT_DIR,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
)
bell_state.generate_device_topology(
    config_root=CONFIG_ROOT,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
    bell_pair=BELL_PAIR,
    output_path=TOPOLOGY_PATH,
)

topology = json.loads(TOPOLOGY_PATH.read_text())
COUPLING_EDGES = [(int(c["control"]), int(c["target"])) for c in topology["couplings"]]
COUPLING_EDGE_SET = set(COUPLING_EDGES)

# Match tmp/4q_spin_chain_qubex: a 1D chain with three interactions.
# For the current Q036-Q039 topology this embeds as 0--1--3--2.
HEISENBERG_CHAIN = (0, 1, 3, 2)
HEISENBERG_EDGES = list(zip(HEISENBERG_CHAIN[:-1], HEISENBERG_CHAIN[1:], strict=True))
if len(REFERENCE_J_VALUES) != len(HEISENBERG_EDGES):
    raise ValueError("REFERENCE_J_VALUES must match the three spin-chain edges")
for edge in HEISENBERG_EDGES:
    if edge not in COUPLING_EDGE_SET and edge[::-1] not in COUPLING_EDGE_SET:
        raise ValueError(f"spin-chain edge {edge} is not available in topology {COUPLING_EDGES!r}")

compile_experiment = bell_state.make_experiment(
    config_root=CONFIG_ROOT,
    device_id=DEVICE_ID,
    qubit_labels=QUBIT_LABELS,
)

def provider_from_experiment(experiment, *, timing_policy: str) -> QubexProvider:
    return QubexProvider.from_experiment(
        experiment,
        name=f"{DEVICE_ID}-4q-{timing_policy}",
        device_topology=TOPOLOGY_PATH,
        qubit_labels=QUBIT_LABELS,
        timing_policy=timing_policy,
        execute_options={
            "state_classification": False,
            "time_integration": True,
            "plot": False,
        },
    )

provider_qiskit = provider_from_experiment(compile_experiment, timing_policy="qiskit")
provider_legacy = provider_from_experiment(compile_experiment, timing_policy="legacy_device_gateway")
backend_qiskit = provider_qiskit.get_backend()
backend_legacy = provider_legacy.get_backend()
backend_ideal = QubexProvider(num_qubits=NUM_QUBITS, coupling_map=COUPLING_EDGES).get_backend()
initial_layout = bell_state.labels_initial_layout(
    qubit_labels=QUBIT_LABELS,
    circuit_labels=QUBIT_LABELS,
)

print("qiskit backend:", backend_qiskit.name)
print("legacy backend:", backend_legacy.name)
print("topology:", TOPOLOGY_PATH)
print("topology svg:", TOPOLOGY_PATH.with_suffix(".svg"))
print("qubits:", QUBIT_LABELS)
print("coupling edges:", COUPLING_EDGES)
print("Heisenberg chain:", HEISENBERG_CHAIN)
print("Heisenberg edges:", HEISENBERG_EDGES)
print("J values:", REFERENCE_J_VALUES)
print("angle scale:", REFERENCE_ANGLE_SCALE)
print("shots:", SHOTS)
print("run on hardware:", RUN_ON_HARDWARE)
print("readout mitigation:", READOUT_MITIGATION)


In [ ]:
display(SVG(filename=str(TOPOLOGY_PATH.with_suffix(".svg"))))


## 4-qubit Heisenberg workload

The model prepares a staggered product state over the topology graph and applies `LAYERS` first-order Heisenberg steps on each calibrated undirected edge. The total angle `theta` is split across layers:

`exp(-i step XX / 2) exp(-i step YY / 2) exp(-i step ZZ / 2)`

The measured observable is the topology-staggered magnetization, normalized to `[-1, 1]`.

In [ ]:
def create_half_ghz_state(qc: QuantumCircuit) -> None:
    ghz_qubits = list(range(NUM_QUBITS // 2))
    qc.h(ghz_qubits[0])
    for control, target in zip(ghz_qubits[:-1], ghz_qubits[1:], strict=True):
        qc.cx(control, target)


def uncompute_half_ghz_state(qc: QuantumCircuit) -> None:
    ghz_qubits = list(range(NUM_QUBITS // 2))
    for control, target in reversed(list(zip(ghz_qubits[:-1], ghz_qubits[1:], strict=True))):
        qc.cx(control, target)
    qc.h(ghz_qubits[0])


def add_xz_pauli_rotation(qc: QuantumCircuit, q0: int, q1: int, angle: float) -> None:
    qc.h(q0)
    qc.rzz(angle, q0, q1)
    qc.h(q0)


def add_heisenberg_interactions(
    qc: QuantumCircuit,
    edges: list[tuple[int, int]],
    j_values: np.ndarray,
    evolution_time: float,
) -> None:
    for (control, target), coupling in zip(edges, j_values, strict=True):
        theta = REFERENCE_ANGLE_SCALE * float(coupling) * evolution_time
        qc.cx(control, target)
        qc.rx(2.0 * theta, control)
        qc.rz(2.0 * theta, target)
        add_xz_pauli_rotation(qc, control, target, -2.0 * theta)
        qc.cx(control, target)


FIRST_HALF_EDGES = HEISENBERG_EDGES[::2]
FIRST_HALF_J_VALUES = REFERENCE_J_VALUES[::2]
SECOND_HALF_EDGES = HEISENBERG_EDGES[1::2]
SECOND_HALF_J_VALUES = REFERENCE_J_VALUES[1::2]


def heisenberg_4q_circuit(total_time: float, *, n_steps: int = N_STEPS) -> QuantumCircuit:
    qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)
    create_half_ghz_state(qc)

    step_time = float(total_time) / n_steps
    add_heisenberg_interactions(qc, FIRST_HALF_EDGES, FIRST_HALF_J_VALUES, step_time / 2.0)
    add_heisenberg_interactions(qc, SECOND_HALF_EDGES, SECOND_HALF_J_VALUES, step_time)
    for _ in range(n_steps - 1):
        add_heisenberg_interactions(qc, FIRST_HALF_EDGES, FIRST_HALF_J_VALUES, step_time)
        add_heisenberg_interactions(qc, SECOND_HALF_EDGES, SECOND_HALF_J_VALUES, step_time)
    add_heisenberg_interactions(qc, FIRST_HALF_EDGES, FIRST_HALF_J_VALUES, step_time / 2.0)

    uncompute_half_ghz_state(qc)
    qc.measure(range(NUM_QUBITS), range(NUM_QUBITS))
    return qc


def probability_zero_from_counts(counts: dict[str, int]) -> float:
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return counts.get("0" * NUM_QUBITS, 0) / total


def exact_probability_zero(circuit: QuantumCircuit) -> float:
    state_circuit = circuit.remove_final_measurements(inplace=False)
    probabilities = Statevector.from_instruction(state_circuit).probabilities_dict()
    return float(probabilities.get("0" * NUM_QUBITS, 0.0))


def compile_physical(total_time: float, backend) -> QuantumCircuit:
    return transpile(
        heisenberg_4q_circuit(total_time),
        backend,
        initial_layout=initial_layout,
        optimization_level=1,
        seed_transpiler=7,
    )


def compile_qiskit(
    total_time: float,
    *,
    scheduling_method: str = "alap",
) -> QuantumCircuit:
    physical = compile_physical(total_time, backend_qiskit)
    return transpile(
        physical,
        backend_qiskit,
        scheduling_method=scheduling_method,
        optimization_level=0,
    )


def compile_legacy(total_time: float) -> QuantumCircuit:
    return compile_physical(total_time, backend_legacy)

print("first half edges:", FIRST_HALF_EDGES, FIRST_HALF_J_VALUES)
print("second half edges:", SECOND_HALF_EDGES, SECOND_HALF_J_VALUES)
heisenberg_4q_circuit(SAMPLE_TOTAL_TIME).draw("text")


## Ideal simulation reference

The local simulator does not model device noise. It gives a reference curve for the hardware measurements.

In [ ]:
ideal = [
    exact_probability_zero(heisenberg_4q_circuit(float(total_time)))
    for total_time in TOTAL_TIMES_IDEAL
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES_IDEAL,
    y=ideal,
    name="ideal simulation",
    line=dict(color="#475569", dash="dot"),
))
fig.update_layout(
    title="4-qubit Heisenberg ideal reference",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="probability of |0000>", range=[-0.05, 1.05]),
    width=760,
    height=430,
)
fig.show()


## Schedule comparison

`qiskit` consumes the Qiskit scheduled circuit, including explicit start times and explicit readout pulses. `legacy_device_gateway` rebuilds the pulse program through the old sequential path and uses `final_measurement=True` at execution time, so its validated pulse schedule excludes the final readout window.

In [ ]:
QISKIT_COMPILE_VARIANTS = {
    "qiskit ALAP": dict(scheduling_method="alap"),
    "qiskit ASAP": dict(scheduling_method="asap"),
}

sample_circuits = {
    name: compile_qiskit(SAMPLE_TOTAL_TIME, **options)
    for name, options in QISKIT_COMPILE_VARIANTS.items()
}
sample_circuits["legacy_device_gateway"] = compile_legacy(SAMPLE_TOTAL_TIME)

sample_schedules = {
    name: backend_qiskit.validate(circuit)[0]
    for name, circuit in sample_circuits.items()
    if name.startswith("qiskit")
}
sample_schedules["legacy_device_gateway"] = backend_legacy.validate(sample_circuits["legacy_device_gateway"])[0]

for name, schedule in sample_schedules.items():
    print(f"{name}")
    print(summarize_pulse_schedule(schedule))

print()
print("qiskit ALAP vs ASAP diff")
print(diff_pulse_schedules(sample_schedules["qiskit ALAP"], sample_schedules["qiskit ASAP"]))

print()
print("qiskit ALAP vs legacy diff")
print(diff_pulse_schedules(sample_schedules["qiskit ALAP"], sample_schedules["legacy_device_gateway"]))


In [ ]:
for name in ("qiskit ALAP", "qiskit ASAP"):
    build_pulse_schedule_timeline_figure(
        sample_schedules[name],
        title=f"4q Heisenberg sample: {name}",
    ).show()


In [ ]:
build_pulse_schedule_timeline_figure(
    sample_schedules["legacy_device_gateway"],
    title="4q Heisenberg sample: legacy_device_gateway",
).show()


In [ ]:
qiskit_circuits_by_variant = {
    name: [compile_qiskit(float(total_time), **options) for total_time in TOTAL_TIMES]
    for name, options in QISKIT_COMPILE_VARIANTS.items()
}
legacy_circuits = [compile_legacy(float(total_time)) for total_time in TOTAL_TIMES]

# Keep the original variable names for quick ad-hoc inspection.
qiskit_circuits = qiskit_circuits_by_variant["qiskit ALAP"]


## Hardware sweep: simulation vs qiskit vs legacy

This cell connects to hardware, builds classifiers once, then runs both circuit batches. `READOUT_MITIGATION=True` applies QUBEX's inverse confusion matrix during provider result conversion. The legacy executor does not emit explicit readout pulses for Qiskit `measure`, so the legacy hardware run sets `final_measurement=True`.

In [ ]:
hardware_sweep = None
if RUN_ON_HARDWARE:
    experiment = bell_state.make_experiment(
        config_root=CONFIG_ROOT,
        device_id=DEVICE_ID,
        qubit_labels=QUBIT_LABELS,
    )
    try:
        experiment.connect()
        execute_provider_qiskit = provider_from_experiment(experiment, timing_policy="qiskit")
        execute_provider_legacy = provider_from_experiment(experiment, timing_policy="legacy_device_gateway")
        execute_provider_qiskit.build_classifier(targets=list(QUBIT_LABELS), shots=SHOTS)
        execute_backend_qiskit = execute_provider_qiskit.get_backend()
        execute_backend_legacy = execute_provider_legacy.get_backend()

        hardware_sweep = {}
        for name, circuits in qiskit_circuits_by_variant.items():
            result = execute_backend_qiskit.run(
                circuits,
                shots=SHOTS,
                readout_mitigation=READOUT_MITIGATION,
            ).result()
            hardware_sweep[name] = [
                probability_zero_from_counts(result.get_counts(i))
                for i in range(len(circuits))
            ]

        legacy_result = execute_backend_legacy.run(
            legacy_circuits,
            shots=SHOTS,
            final_measurement=True,
            readout_mitigation=READOUT_MITIGATION,
        ).result()
        hardware_sweep["legacy_device_gateway"] = [
            probability_zero_from_counts(legacy_result.get_counts(i))
            for i in range(len(legacy_circuits))
        ]
        print(hardware_sweep)
    finally:
        experiment.disconnect()
else:
    print("RUN_ON_HARDWARE is False; hardware sweep was skipped.")


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES_IDEAL,
    y=ideal,
    name="ideal simulation",
    line=dict(color="#475569", dash="dot"),
))
if hardware_sweep is not None:
    for name, values in hardware_sweep.items():
        fig.add_trace(go.Scatter(
            x=TOTAL_TIMES,
            y=values,
            mode="lines+markers",
            name=f"{name} hardware",
        ))
fig.update_layout(
    title="4-qubit Heisenberg: simulation vs hardware timing policies",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="probability of |0000>", range=[-0.05, 1.05]),
    width=760,
    height=430,
)
fig.show()
